# 10 — Block 1: Data management

Chia tập, lập index ảnh (không nhãn) và tính sẵn embedding cho pipeline proposal (`src/rav`, cùng contract với P-026).

| Block | Notebook | Vào | Ra |
|---|---|---|---|
| 1. Data management | `10_data_management` | BDD100K | `splits/`, `pool/*_index.csv`, `fields/` (embedding cache) |
| 2. Proposal nhiều vòng | `20_proposal_rounds` | output 10 | `proposals/<cách>/s<seed>_k<K>/export.csv` (+ history, meta) |
| 3. Finetune + đánh giá | `30_finetune_eval` | output 10 + 20 (hoặc export.csv từ P-026) | `runs/`, `eval/report.md` |
| 4. Hiển thị + lọc | `40_explore` | output 10 (+ 20, 30) | bảng lọc, lưới ảnh, lỗi model |

**Input:** BDD100K (`solesensei_bdd100k`). **GPU:** nên có (embedding). **Output:** `art/splits/`, `art/pool/`, `art/fields/`.

In [ ]:
# === Setup (giống nhau ở mọi notebook) ===
SMOKE = True   # True: pipeline thật trên tập nhỏ. Run thật: False
SCENARIO = "missing"   # missing: base chưa có novel class | weak: base có ít ảnh novel
REPO_URL = "https://github.com/Le-Anh-Duy/topicx-yolo-expansion.git"
COMMIT = "main"   # run thật: pin commit hash
import os, subprocess, sys
R = "/tmp/repo"
if not os.path.exists(R):
    subprocess.run(["git", "clone", "-q", REPO_URL, R], check=True)
# luôn lấy bản mới nhất của COMMIT từ GitHub (kể cả khi /tmp/repo còn từ lần chạy trước trong session tương tác)
subprocess.run(["git", "-C", R, "fetch", "-q", "origin", COMMIT], check=True)
subprocess.run(["git", "-C", R, "checkout", "-q", "-f", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{R}/requirements.txt"], check=True)
for p in (f"{R}/src", R):   # `topicx` (harness) và `src.rav` (pipeline proposal, cùng đường import với P-026)
    if p not in sys.path:
        sys.path.insert(0, p)
for m in [m for m in sys.modules if m.split(".")[0] in ("topicx", "src")]:
    del sys.modules[m]   # chạy lại cell thì nạp lại code mới
import numpy as np, pandas as pd
from topicx import common as C, pipeline as P, proposals as PR
cfg = C.load_config(SMOKE, SCENARIO)
commit = subprocess.run(["git", "-C", R, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print("code:", commit)
print(f"kịch bản={SCENARIO} | artifact={cfg['art']} | copy từ input: {C.sync_inputs(cfg)} file")

In [ ]:
STAGE = "data"
# Test (CPU, vài giây): protocol của harness + test lõi P-026 + component mới. Lỗi thì dừng notebook.
subprocess.run([sys.executable, "-m", "pytest", "-q", f"{R}/tests"], check=True)
C.env_report(cfg, STAGE);

## Health check dataset
Vài giây, không đọc cả file nhãn. Có mục FAIL thì notebook dừng ở đây.

In [ ]:
hc, sample = P.health_check(cfg)
if sample is not None:
    display(sample.resize((640, 360)))   # box đỏ phải ôm đúng vật thể

In [ ]:
NOVEL, REASON = "bus", "EDA / giả thuyết"   # class mục tiêu; phải khớp ở mọi notebook sau
RUN_EDA = False                                # True: in bảng EDA (đọc cả file nhãn ~1 phút) trước khi chia tập

In [ ]:
if RUN_EDA and not (C.WORK / cfg["art"] / "splits/meta.json").exists():
    images, boxes, paths = P.load_bdd(cfg)
    for name, t in P.eda(cfg, images, boxes).items():
        print(name); display(t)

## Chia tập (base / dev / pool / test), oracle giữ nhãn pool

In [ ]:
meta = P.prepare_splits(cfg, NOVEL, REASON)
display(pd.DataFrame(meta["report"]).T)
print("novel instances ở final test:", meta["test_novel_instances"])

## Index + embedding cho pipeline proposal

Source `bdd_images` của `src/rav` đọc `pool/<split>_index.csv` (tên ảnh + đường dẫn, **không nhãn**); unit id dạng P-026 `"<tên ảnh>:0.000-0.000"`.
Field được cache ở `art/fields/` (một file / field). Notebook 20 gắn output này thì không tính lại.

In [ ]:
for split in ("pool", "dev"):
    print(PR.write_index(cfg, split))
print(PR.prepare_fields(cfg, "pool"))                               # emb.dinov2_s, emb.clip_b32x3, emb.clip_b32
print(PR.prepare_fields(cfg, "dev", fields=("emb.clip_b32x3",)))   # để chọn câu topic / τ trên dev ở notebook 20

## Gói kết quả
Zip file json/csv/md nhỏ (không có weights, embedding, nhãn oracle) → tab Output.

In [ ]:
P.bundle(cfg, STAGE)